# 🇧🇩 "Amar Tax" AI Agent - Virtual Tax Advisor
### Applied AI with Multi-Agent Systems (CrewAI) | Intermediate Level

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1UfgNv61I-ICv6S-iWflV8yC0FJtt1HLq)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/Udipto-Mondal/Amar-Tax-AI-Agent.git)

---

## 📌 Project Overview
Filing an individual income tax return in Bangladesh confuses many people: who must file, what the tax-free threshold is, how the investment rebate is applied, and which documents are required.

This project builds an autonomous **Multi-Agent System (MAS)** using **CrewAI** that acts as a **"Virtual Tax Advisor"**.
The system takes a taxpayer profile (**Age, Gender, Profession, Income, Investment, Location, TIN status**) and outputs a comprehensive **Tax Readiness Report** featuring:
1. **English Markdown Summary Table**
2. **Audit / Edge Case Warning Flags**
3. **3–4 line Fluent Bangla Summary**

---

## 👥 The Crew Definition (4 Specialized Agents)
| # | Agent Role | Goal & Responsibilities | Tools Used |
|---|---|---|---|
| **1** | **Senior Taxpayer Classification Officer** | Determines mandatory filing status, taxpayer category, tax-free threshold, and regional minimum tax. Flags minor status. | `lookup_tax_db` |
| **2** | **Chartered Tax Accountant** | Calculates slab-based gross tax, allowable investment cap (20%), 15% investment rebate, and net payable tax vs minimum tax. | `lookup_tax_db`, `calculate_slab_tax` |
| **3** | **Return Filing Documentation Officer** | Compiles tailored checklist of certificates and proofs based on profession and income sources. Places TIN Registration at top if no TIN. | `get_document_checklist` |
| **4** | **Compliance & Audit Reviewer** | Cross-audits all outputs, catches all 6 edge cases, assigns `✅ Passed` or `⚠️ Flagged`, and formats final report with Bangla summary. | `validate_compliance_and_edge_cases` |

---

## 🔄 Architecture & Execution Pipeline
```
User Profile ──▶ [Agent 1: Classifier]  ──(threshold, min_tax)──▶ [Agent 2: Slab Calculator]
                       │                                                    │
                       ▼                                                    ▼
            [Agent 3: Document Organizer]                           (tax computation)
                       │                                                    │
                       └──────────────────▶ [Agent 4: Reviewer] ◀──────────┘
                                                    │
                                                    ▼
                                  [English Markdown Table + Bangla Summary]
```

## 📦 Step 1: Install Dependencies
Run the cell below to install CrewAI and Google GenAI libraries in the Colab runtime.

In [1]:
# Install required dependencies
!pip install -q crewai crewai-tools pydantic python-dotenv google-genai

## 🔑 Step 2: Configure API Key
You can configure your **Google Gemini API Key** or **Groq / OpenAI API Key** below.

In [2]:
import os
import getpass

# Try retrieving from Google Colab userdata secrets
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    if api_key:
        os.environ['GEMINI_API_KEY'] = api_key
except Exception:
    pass

if not os.environ.get('GEMINI_API_KEY') and not os.environ.get('GROQ_API_KEY'):
    print('Please enter your Gemini API Key (or set it in Colab Secrets):')
    key = getpass.getpass('GEMINI_API_KEY: ')
    if key:
        os.environ['GEMINI_API_KEY'] = key.strip()

print('API Key successfully configured!')

API Key successfully configured!


## 🗄️ Step 3: Local Fallback Database (Source of Truth)
This database contains the simulated 2026 tax year parameters from the assignment specification.
**Requirement 7**: If remote scraping/API fails, the system logs a fallback message and loads this Local DB.

In [3]:
import json
import os
from pathlib import Path

TAX_DB_DATA = {
    "tax_year": "2026-27",
    "filing_deadline": "2026-11-30",
    "thresholds": {
        "general": 350000,
        "female": 400000,
        "senior_65_plus": 400000,
        "disabled": 475000,
        "freedom_fighter": 500000
    },
    "slabs_after_threshold": [
        {"limit": 100000, "rate": 0.05},
        {"limit": 400000, "rate": 0.10},
        {"limit": 500000, "rate": 0.15},
        {"limit": 500000, "rate": 0.20},
        {"limit": None, "rate": 0.25}
    ],
    "minimum_tax": {
        "dhaka_chattogram_city": 5000,
        "other_city_corporation": 4000,
        "rest_of_country": 3000
    },
    "investment_rebate": {
        "rate": 0.15,
        "max_investment_percent_of_income": 0.20
    },
    "required_docs": {
        "common": ["TIN Certificate", "NID Copy", "Bank Statement"],
        "salaried": ["Salary Certificate", "Employer Details"],
        "business": ["Trade License", "Profit & Loss Statement", "Balance Sheet"],
        "freelancer": ["Foreign Remittance Proof", "Client Invoices"],
        "landlord": ["Rent Agreement", "Municipal Tax Receipt"],
        "investor": ["Investment Certificates (DPS/Sanchaypatra)", "Bank Interest Certificate"]
    }
}

# Save to data/tax_db.json
os.makedirs('data', exist_ok=True)
with open('data/tax_db.json', 'w', encoding='utf-8') as f:
    json.dump(TAX_DB_DATA, f, indent=2)

print('Local Fallback Database created at data/tax_db.json!')

Local Fallback Database created at data/tax_db.json!


## 🧩 Step 4: Database Loader with Network Fallback
Demonstrates the graceful network fallback pattern required by Requirement 7.

In [4]:
import logging
import urllib.request
import urllib.error
from typing import Any, Dict, List, Optional, Tuple

logger = logging.getLogger('AmarTax')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

LOCAL_DB_PATH = Path('data/tax_db.json')
REMOTE_API_URL = 'https://nbr.gov.bd/api/v1/tax_rules_2026.json'

def load_tax_database(source_url: Optional[str] = None) -> Dict[str, Any]:
    """Requirement 7: Try remote fetch, except fallback to local DB with log message."""
    target_url = source_url or REMOTE_API_URL
    try:
        req = urllib.request.Request(target_url, headers={'User-Agent': 'AmarTax-MAS/1.0'})
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                return json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        logger.warning(f'[FALLBACK TRIGGERED] Remote tax API failed: {e}. Gracefully falling back to Local DB: {LOCAL_DB_PATH}')

    with open(LOCAL_DB_PATH, 'r', encoding='utf-8') as f:
        return json.load(f)

def resolve_taxpayer_category(age: int, gender: str, is_disabled: bool = False, is_freedom_fighter: bool = False) -> Tuple[str, str]:
    if is_freedom_fighter:
        return 'freedom_fighter', 'Freedom Fighter'
    if is_disabled:
        return 'disabled', 'Disabled'
    if age >= 65:
        return 'senior_65_plus', 'Senior 65+'
    if gender.strip().lower() in ['female', 'woman', 'f']:
        return 'female', 'Female'
    return 'general', 'General'

def get_tax_threshold(category_key: str, db: Dict[str, Any]) -> float:
    return float(db.get('thresholds', {}).get(category_key, 350000))

def resolve_minimum_tax(location: str, db: Dict[str, Any]) -> Tuple[float, str]:
    loc = location.strip().lower()
    min_taxes = db.get('minimum_tax', {})
    if any(c in loc for c in ['dhaka', 'chattogram', 'chittagong']):
        return float(min_taxes.get('dhaka_chattogram_city', 5000)), 'Dhaka/Chattogram City Corporation'
    other_cities = ['gazipur', 'narayanganj', 'rajshahi', 'khulna', 'sylhet', 'barishal', 'rangpur', 'mymensingh', 'cumilla']
    if any(c in loc for c in other_cities) or 'city corporation' in loc:
        return float(min_taxes.get('other_city_corporation', 4000)), 'Other City Corporation'
    return float(min_taxes.get('rest_of_country', 3000)), 'Rest of Country'

def calculate_tax_breakdown(income: float, threshold: float, actual_investment: float, minimum_tax: float, db: Dict[str, Any]) -> Dict[str, Any]:
    slabs = db.get('slabs_after_threshold', [])
    rebate_rules = db.get('investment_rebate', {'rate': 0.15, 'max_investment_percent_of_income': 0.20})
    rebate_rate = rebate_rules.get('rate', 0.15)
    max_inv_pct = rebate_rules.get('max_investment_percent_of_income', 0.20)

    if income <= threshold:
        allowable_inv = min(max(0.0, actual_investment), max_inv_pct * max(0.0, income))
        return {
            'total_income': income,
            'tax_free_threshold': threshold,
            'taxable_after_threshold': 0.0,
            'slab_breakdown': ['Income is within tax-free threshold; 0% tax applies.'],
            'gross_tax': 0.0,
            'allowable_investment': allowable_inv,
            'investment_rebate': 0.0,
            'minimum_tax': minimum_tax,
            'net_payable_tax': 0.0,
            'income_above_threshold': False
        }

    taxable_amount = income - threshold
    remaining = taxable_amount
    gross_tax = 0.0
    slab_breakdown = []

    for slab in slabs:
        limit = slab.get('limit')
        rate = slab.get('rate', 0.0)
        pct = int(rate * 100)
        if limit is None:
            if remaining > 0:
                tax = remaining * rate
                gross_tax += tax
                slab_breakdown.append(f'Remaining {remaining:,.0f} BDT @ {pct}% = {tax:,.0f} BDT')
                remaining = 0.0
            break
        else:
            in_slab = min(remaining, float(limit))
            if in_slab > 0:
                tax = in_slab * rate
                gross_tax += tax
                slab_breakdown.append(f'Next {in_slab:,.0f} BDT @ {pct}% = {tax:,.0f} BDT')
                remaining -= in_slab
            if remaining <= 0:
                break

    allowable_inv = min(max(0.0, actual_investment), max_inv_pct * income)
    rebate = allowable_inv * rebate_rate
    tax_after_rebate = max(0.0, gross_tax - rebate)
    net_payable_tax = max(tax_after_rebate, minimum_tax)

    return {
        'total_income': income,
        'tax_free_threshold': threshold,
        'taxable_after_threshold': taxable_amount,
        'slab_breakdown': slab_breakdown,
        'gross_tax': gross_tax,
        'allowable_investment': allowable_inv,
        'investment_rebate': rebate,
        'minimum_tax': minimum_tax,
        'net_payable_tax': net_payable_tax,
        'income_above_threshold': True
    }

def get_required_documents(profession: str, income_sources: Optional[List[str]] = None, has_tin: bool = True, has_investment: bool = False, db: Optional[Dict[str, Any]] = None) -> Tuple[List[str], bool]:
    database = db or load_tax_database()
    req_docs = database.get('required_docs', {})
    common_docs = list(req_docs.get('common', ['TIN Certificate', 'NID Copy', 'Bank Statement']))
    checklist = []

    # Edge Case 5: Missing TIN -> TIN Registration at top
    if not has_tin:
        checklist.append('TIN Registration')
        common_docs = [d for d in common_docs if d != 'TIN Certificate']

    prof_clean = profession.strip().lower()
    sources = [s.strip().lower() for s in (income_sources or [])]
    doc_keys_found = []

    for key in ['salaried', 'business', 'freelancer', 'landlord', 'investor']:
        if key in prof_clean or any(key in s for s in sources):
            doc_keys_found.append(key)

    if any(k in prof_clean or any(k in s for s in sources) for k in ['engineer', 'officer', 'manager', 'employee', 'salary']):
        if 'salaried' not in doc_keys_found:
            doc_keys_found.append('salaried')
    if any(k in prof_clean or any(k in s for s in sources) for k in ['freelance', 'remittance']):
        if 'freelancer' not in doc_keys_found:
            doc_keys_found.append('freelancer')
    if has_investment or any(k in prof_clean or any(k in s for s in sources) for k in ['dps', 'sanchaypatra', 'invest']):
        if 'investor' not in doc_keys_found:
            doc_keys_found.append('investor')

    is_general_fallback = False
    if not doc_keys_found:
        is_general_fallback = True
        checklist.extend(common_docs)
    else:
        checklist.extend(common_docs)
        for key in doc_keys_found:
            for item in req_docs.get(key, []):
                if item not in checklist:
                    checklist.append(item)

    deduped = []
    for item in checklist:
        if item not in deduped:
            deduped.append(item)
    return deduped, is_general_fallback

## 📐 Step 5: Pydantic Structured Output Models
**Requirement 4**: Use a Pydantic model (`output_pydantic`) in at least one Task.

In [5]:
from pydantic import BaseModel, Field

class TaxpayerEligibilityOutput(BaseModel):
    taxpayer_category: str = Field(..., description='Category: Female, General, Senior 65+, Disabled, Freedom Fighter')
    is_mandatory: bool = Field(..., description='Whether return filing is mandatory')
    tax_free_threshold: float = Field(..., description='Tax-free threshold amount in BDT')
    minimum_tax: float = Field(..., description='Location minimum tax amount in BDT')
    location_tier: str = Field(..., description='Matched location jurisdiction')
    eligibility_notes: str = Field(..., description='Notes on filing criteria and minor restrictions')
    flags: List[str] = Field(default_factory=list, description='Audit flags')

class TaxCalculationOutput(BaseModel):
    total_income: float = Field(..., description='Total annual income in BDT')
    tax_free_threshold: float = Field(..., description='Tax-free threshold in BDT')
    taxable_income_after_threshold: float = Field(..., description='Taxable income in BDT')
    gross_tax: float = Field(..., description='Gross tax in BDT')
    slab_breakdown: List[str] = Field(default_factory=list, description='Slab calculations')
    allowable_investment: float = Field(..., description='Allowable investment in BDT')
    investment_rebate: float = Field(..., description='Investment rebate in BDT')
    minimum_tax: float = Field(..., description='Minimum tax in BDT')
    net_payable_tax: float = Field(..., description='Net tax payable in BDT')
    calculation_summary: str = Field(..., description='Explanation of calculation')
    flags: List[str] = Field(default_factory=list, description='Calculation flags')

class DocumentChecklistOutput(BaseModel):
    profession: str
    has_tin: bool
    documents_needed: List[str]
    is_fallback_checklist: bool
    notes: str = ''

## 🛠️ Step 6: Custom Tools (@tool)
**Requirement 3**: Build at least one `@tool` (`lookup_tax_db`, `calculate_slab_tax`, `get_document_checklist`, `validate_compliance_and_edge_cases`).

In [6]:
from crewai.tools import tool

@tool('lookup_tax_db')
def lookup_tax_db(query_type: str, category: str = '', location: str = '') -> str:
    """Query the tax database for thresholds, minimum taxes, slabs, and document rules."""
    db = load_tax_database()
    query = query_type.strip().lower()
    if query == 'thresholds' and category:
        cat_key = category.strip().lower().replace(' ', '_')
        return json.dumps({'category': category, 'tax_free_threshold': get_tax_threshold(cat_key, db)}, indent=2)
    if query == 'minimum_tax' and location:
        min_tax, tier = resolve_minimum_tax(location, db)
        return json.dumps({'location': location, 'matched_tier': tier, 'minimum_tax': min_tax}, indent=2)
    return json.dumps(db, indent=2)

@tool('calculate_slab_tax')
def calculate_slab_tax(income: float, threshold: float, investment: float, minimum_tax: float) -> str:
    """Computes progressive slab taxes, investment rebate (15% capped at 20% of income), and net payable tax."""
    db = load_tax_database()
    res = calculate_tax_breakdown(float(income), float(threshold), float(investment), float(minimum_tax), db)
    return json.dumps(res, indent=2)

@tool('get_document_checklist')
def get_document_checklist(profession: str, income_sources_csv: str = '', has_tin: bool = True, has_investment: bool = False) -> str:
    """Generates document checklist. Places TIN Registration at top if no TIN; falls back to general list if profession not in DB."""
    db = load_tax_database()
    sources = [s.strip() for s in income_sources_csv.split(',') if s.strip()]
    checklist, is_fallback = get_required_documents(profession, sources, has_tin, has_investment, db)
    return json.dumps({'profession': profession, 'has_tin': has_tin, 'is_general_fallback': is_fallback, 'documents_needed': checklist}, indent=2)

@tool('validate_compliance_and_edge_cases')
def validate_compliance_and_edge_cases(age: int, income: float, investment: float, threshold: float, claims_required_to_file: bool = False, has_tin: bool = True, wants_independent_filing: bool = True, profession_is_fallback: bool = False) -> str:
    """Validates all 6 required edge cases and assigns compliance status."""
    flags = []
    if age < 18 and wants_independent_filing:
        flags.append('⚠️ Minor cannot file independently')
    if income < 0:
        flags.append('⚠️ Flag as invalid input: Income cannot be negative')
    if investment > income and income > 0:
        flags.append('⚠️ Investment exceeds income, please verify')
    if income <= threshold and claims_required_to_file:
        flags.append(f'⚠️ Inform them that filing is not mandatory (income {income:,.0f} BDT is within threshold {threshold:,.0f} BDT, but check Minimum Tax rule)')
    if not has_tin:
        flags.append("⚠️ No TIN: 'TIN Registration' must be added at top of checklist")
    if profession_is_fallback:
        flags.append('⚠️ Profession not in DB: Fall back to general checklist')

    status = '⚠️ Flagged' if flags else '✅ Passed'
    return json.dumps({'compliance_status': status, 'total_flags': len(flags), 'flags': flags}, indent=2)

## 🤖 Step 7: Define The 4 Specialized Agents
**Requirement 1**: Every agent must have a role, goal, and backstory.
**Requirement 8**: Use `verbose=True`.

In [7]:
from crewai import Agent, LLM

# Initialize CrewAI LLM with Gemini 3.5 Flash Lite
gemini_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
llm = LLM(model='gemini/gemini-3.5-flash-lite', api_key=gemini_key, temperature=0.1)

classifier_agent = Agent(
    role='Senior Taxpayer Classification Officer',
    goal='Determine mandatory filing status, taxpayer category, tax-free threshold, and regional minimum tax based on profile. Flag minors under 18.',
    backstory='Senior Taxpayer Classification Officer with over 15 years at the National Board of Revenue (NBR) in Bangladesh. Strict enforcer of statutory taxpayer categories and legal age rules.',
    tools=[lookup_tax_db],
    llm=llm,
    verbose=True
)

calculator_agent = Agent(
    role='Chartered Tax Accountant',
    goal='Using the Classifier context, calculate slab-based gross tax, allowable investment (capped at 20%), 15% rebate, and net payable tax vs minimum tax. Never invent rates.',
    backstory='Fellow Chartered Accountant (FCA) specializing in Bangladesh individual income tax calculation. Renowned for arithmetic perfection and strict adherence to the Local Fallback Database.',
    tools=[lookup_tax_db, calculate_slab_tax],
    llm=llm,
    verbose=True
)

organizer_agent = Agent(
    role='Return Filing Documentation Officer',
    goal='Generate customized document checklist. If no TIN, mandate TIN Registration at top. If profession not in DB, fall back to general list.',
    backstory='Seasoned Return Filing Documentation Officer at the NBR Central Service Center, dedicated to ensuring taxpayers submit all necessary legal certificates to avoid penalties.',
    tools=[get_document_checklist],
    llm=llm,
    verbose=True
)

reviewer_agent = Agent(
    role='Compliance & Audit Reviewer',
    goal='Cross-check outputs of other three agents, detect all edge cases, assign final status of Passed or Flagged, and generate English Markdown Table + 3-4 line Bangla summary.',
    backstory='Chief Compliance & Audit Reviewer at the NBR. Possesses an unforgiving eye for data inconsistencies, calculation discrepancies, and statutory edge cases.',
    tools=[validate_compliance_and_edge_cases],
    llm=llm,
    verbose=True
)

## 📋 Step 8: Define Tasks & Context Passing
**Requirement 2**: Classifier ➔ Calculator ➔ Organizer ➔ Reviewer; pass outputs using `context=[...]`.

In [8]:
from crewai import Task, Crew, Process

def build_tax_crew(profile: dict) -> Crew:
    task1 = Task(
        description=(
            f"Analyze profile: Age={profile.get('age')}, Gender={profile.get('gender')}, "
            f"Profession={profile.get('profession')}, Total Income={profile.get('total_income'):,} BDT, "
            f"Location={profile.get('location')}, Has TIN={profile.get('has_tin')}, "
            f"Claims Required={profile.get('claims_required_to_file')}, Wants Independent Filing={profile.get('wants_independent_filing')}.\n"
            "1. Use `lookup_tax_db` to determine category, threshold, and minimum tax.\n"
            "2. CRITICAL RULE: If age < 18 and wants independent filing, flag '⚠️ Minor cannot file independently'."
        ),
        expected_output="Structured TaxpayerEligibilityOutput with category, threshold, minimum tax, notes, flags.",
        agent=classifier_agent,
        output_pydantic=TaxpayerEligibilityOutput
    )

    task2 = Task(
        description=(
            f"Using Classifier context, compute tax for total income {profile.get('total_income'):,} BDT and "
            f"investment {profile.get('investment'):,} BDT.\n"
            "1. Use `calculate_slab_tax` and `lookup_tax_db` strictly by Local DB rules.\n"
            "2. Slabs begin after threshold. Allowable investment = min(actual, 20% * income). Rebate = 15% * allowable.\n"
            "3. Net tax = max(Gross - Rebate, Minimum Tax) when income > threshold. If income <= threshold, gross=0, net=0.\n"
            "4. Flag if investment > income."
        ),
        expected_output="Structured TaxCalculationOutput with gross tax, rebate, minimum tax, and net payable tax.",
        agent=calculator_agent,
        context=[task1],
        output_pydantic=TaxCalculationOutput
    )

    task3 = Task(
        description=(
            f"Generate document checklist for profession '{profile.get('profession')}', sources '{profile.get('income_sources')}', "
            f"has_tin={profile.get('has_tin')}, investment={profile.get('investment'):,} BDT.\n"
            "1. If no TIN, add 'TIN Registration' at top of checklist.\n"
            "2. If profession not in DB, fall back to general checklist."
        ),
        expected_output="Structured DocumentChecklistOutput with required documents list.",
        agent=organizer_agent,
        context=[task1],
        output_pydantic=DocumentChecklistOutput
    )

    task4 = Task(
        description=(
            f"Audit and cross-check outputs from Classifier, Calculator, and Documentation Officer.\n"
            "1. Use `validate_compliance_and_edge_cases` to verify all 6 edge cases.\n"
            "2. Assign final compliance status: '✅ Passed' or '⚠️ Flagged'.\n"
            "3. Output:\n"
            "   A) English Markdown Table with fields: Taxpayer Category, Total Income, Tax-free Threshold, "
            "      Gross Tax (Slab-based), Investment Rebate (15%), Minimum Tax (<Location>), Net Payable Tax, "
            "      Documents Needed, Compliance Status.\n"
            "   B) Warning / Flag Section (if any flags).\n"
            "   C) Exactly 3-4 lines Bangla Summary."
        ),
        expected_output="Comprehensive Tax Readiness Report with Markdown Table, Warning flags, and Bangla summary.",
        agent=reviewer_agent,
        context=[task1, task2, task3]
    )

    return Crew(
        agents=[classifier_agent, calculator_agent, organizer_agent, reviewer_agent],
        tasks=[task1, task2, task3, task4],
        process=Process.sequential,
        verbose=True
    )

def run_tax_advisory(profile: dict):
    crew = build_tax_crew(profile)
    return str(crew.kickoff())

## 🧪 Step 9: Test Case 1 - The Example Scenario (Baseline Acceptance)
**Input**:
> *"I am a 34-year-old woman working as a software engineer, and I live in Dhaka. My annual salary is 1,200,000 BDT and I earn 300,000 BDT from freelancing (remittance comes through the bank). I have invested 200,000 BDT in DPS and Sanchaypatra. I have a TIN."*

**Expected Output**:
- Net Tax = **110,000 BDT**
- Compliance Status = **✅ Passed**

In [9]:
from IPython.display import display, Markdown

tc1_profile = {
    'age': 34,
    'gender': 'Female',
    'profession': 'Software Engineer',
    'total_income': 1500000.0,
    'investment': 200000.0,
    'location': 'Dhaka',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['salaried', 'freelancer', 'investor']
}

report1 = run_tax_advisory(tc1_profile)
display(Markdown(report1))

### Final Tax Readiness Report

#### A) Executive Audit Table

| Field | Result |
| :--- | :--- |
| Taxpayer Category | Female |
| Total Income | 1,500,000.0 BDT |
| Tax-free Threshold | 400,000.0 BDT |
| Gross Tax (Slab-based) | 140,000.0 BDT |
| Investment Rebate (15%) | 30,000.0 BDT |
| Minimum Tax (Dhaka City Corporation) | 5,000.0 BDT |
| Net Payable Tax | 110,000.0 BDT |
| Documents Needed | TIN Certificate, NID Copy, Bank Statement, Salary Certificate, Employer Details, Foreign Remittance Proof, Client Invoices, Investment Certificates (DPS/Sanchaypatra), Bank Interest Certificate |
| Compliance Status | ✅ Passed |

#### B) Warning / Flag Section
*No compliance anomalies, missing prerequisites, or contradictory declarations were detected during the audit. All inputs are valid, and the calculation strictly adheres to the governing tax schedules.*

#### C) Bangla Summary (বাংলা সারসংক্ষেপ)
আপনার বার্ষিক মোট আয় ১৫,০০,০০০ টাকা এবং নারীদের জন্য করমুক্ত সীমার পরিমাণ ৪,০০,০০০ টাকা। সকল হিসাব ও বিনিয়োগ রেয়াত (৩০,০০০ টাকা) বাদ দেওয়ার পর আপনার নিট প্রদেয় করের পরিমাণ দাঁড়ায় ১,১০,০০০ টাকা। যেহেতু আপনার আয় করমুক্ত সীমার উপরে, তাই সময়মতো আয়কর রিটার্ন দাখিল করা বাধ্যতামূলক।


## 🧪 Step 10: Test Case 2 - Minor Independent Filing (Edge Case 1)
**Input**: 16-year-old, Income 500,000 BDT, wants to file independently.
**Expected Behavior**: `⚠️ Minor cannot file independently` flag raised.

In [10]:
tc2_profile = {
    'age': 16,
    'gender': 'Male',
    'profession': 'Freelancer',
    'total_income': 500000.0,
    'investment': 0.0,
    'location': 'Dhaka',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['freelancer']
}

report2 = run_tax_advisory(tc2_profile)
display(Markdown(report2))

### Tax Readiness Report

| Field | Result |
| :--- | :--- |
| Taxpayer Category | General |
| Total Income | 500,000.0 BDT |
| Tax-free Threshold | 350,000.0 BDT |
| Gross Tax (Slab-based) | 10,000.0 BDT |
| Investment Rebate (15%) | 0.0 BDT |
| Minimum Tax (Dhaka City Corporation) | 5,000.0 BDT |
| Net Payable Tax | 10,000.0 BDT |
| Documents Needed | TIN Certificate, NID Copy, Bank Statement, Foreign Remittance Proof, Client Invoices |
| Compliance Status | ⚠️ Flagged |

### Compliance Warning & Flags
- ⚠️ **Minor cannot file independently**: The taxpayer is 16 years old. Under NBR regulations, a minor under 18 years of age cannot file an independent tax return.

---

### Bangla Summary (বাংলা সারসংক্ষেপ)
আপনার বার্ষিক মোট আয় ৫,০০,০০০ টাকা এবং করমুক্ত সীমার পরিমাণ ৩,৫০,০০০ টাকা। সকল হিসাব অনুযায়ী আপনার নিট প্রদেয় কর ১০,০০০ টাকা নির্ধারিত হয়েছে। তবে আপনার বয়স ১৬ বছর হওয়ায় জাতীয় রাজস্ব বোর্ডের নিয়ম অনুযায়ী আপনি নিজে স্বাধীনভাবে রিটার্ন দাখিল করতে পারবেন না।


## 🧪 Step 11: Test Case 3 - Below Tax-Free Threshold in Rangpur (Edge Case 4)
**Input**: 45-year-old male, Income 300,000 BDT in Rangpur, claims required to file.
**Expected Behavior**: Filing not mandatory, with an explanatory warning message.

In [11]:
tc3_profile = {
    'age': 45,
    'gender': 'Male',
    'profession': 'Teacher',
    'total_income': 300000.0,
    'investment': 0.0,
    'location': 'Rangpur',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': True,
    'income_sources': ['salaried']
}

report3 = run_tax_advisory(tc3_profile)
display(Markdown(report3))

### Executive Tax Readiness Report

| Field | Result |
| :--- | :--- |
| Taxpayer Category | General |
| Total Income | 300,000.0 BDT |
| Tax-free Threshold | 350,000.0 BDT |
| Gross Tax (Slab-based) | 0.0 BDT |
| Investment Rebate (15%) | 0.0 BDT |
| Minimum Tax (Rest of Country) | 3,000.0 BDT |
| Net Payable Tax | 0.0 BDT |
| Documents Needed | TIN Certificate, NID Copy, Bank Statement, Salary Certificate, Employer Details |
| Compliance Status | ⚠️ Flagged |

---

### Compliance & Edge Case Audit Findings
- **⚠️ Flagged**: Your total annual income of 300,000.0 BDT is below the statutory tax-free threshold of 350,000.0 BDT for General taxpayers. Although you claimed that filing is required, income tax return submission is technically not mandatory for you based on income alone (though holding a TIN carries certain compliance requirements). Please verify if you fall under any other mandatory criteria (such as owning a motor car, asset threshold, etc.).

---

### নাগরিক সারাংশ (Bangla Summary)
আপনার বার্ষিক মোট আয় ৩,০০,০০০ টাকা, যা সাধারণ করদাতার জন্য নির্ধারিত ৩,৫০,০০০ টাকার কর-মুক্ত সীমার নিচে রয়েছে। যেহেতু আপনার আয় কর-মুক্ত সীমার নিচে, তাই আপনার কোনো নিট প্রদেয় কর বা আয়কর রিটার্ন জমা দেওয়া বাধ্যতামূলক নয় (যদি না অন্য কোনো বিশেষ শর্ত প্রযোজ্য হয়)। আপনার সুবিধার জন্য প্রয়োজনীয় নথিপত্র প্রস্তুত রাখা হয়েছে, তবে বর্তমান আয়ের ভিত্তিতে কর রিটার্ন দাখিল বাধ্যতামূলক নয়।


## 🧪 Step 12: Test Case 4 - Excessive Investment (Edge Case 3)
**Input**: Income 800,000 BDT, Investment 1,000,000 BDT.
**Expected Behavior**: `⚠️ Investment exceeds income, please verify` flag raised.

In [12]:
tc4_profile = {
    'age': 40,
    'gender': 'Male',
    'profession': 'Business',
    'total_income': 800000.0,
    'investment': 1000000.0,
    'location': 'Chattogram',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['business', 'investor']
}

report4 = run_tax_advisory(tc4_profile)
display(Markdown(report4))

### Final Tax Readiness Report

| Field | Result |
| :--- | :--- |
| Taxpayer Category | General |
| Total Income | 800,000.0 BDT |
| Tax-free Threshold | 350,000.0 BDT |
| Gross Tax (Slab-based) | 40,000.0 BDT |
| Investment Rebate (15%) | 24,000.0 BDT |
| Minimum Tax (Dhaka/Chattogram City Corporation) | 5,000.0 BDT |
| Net Payable Tax | 16,000.0 BDT |
| Documents Needed | TIN Certificate, NID Copy, Bank Statement, Trade License, Profit & Loss Statement, Balance Sheet, Investment Certificates (DPS/Sanchaypatra), Bank Interest Certificate |
| Compliance Status | ⚠️ Flagged |

---

### Warning / Flag Section
- **⚠️ Investment exceeds income, please verify**: The declared investment (1,000,000.0 BDT) is greater than the total annual income (800,000.0 BDT). Please verify these financial figures as allowable investment limits and documentation may be subject to further scrutiny.

---

### Bangla Summary (বাংলা সারসংক্ষেপ)
আপনার বার্ষিক মোট আয় ৮,০০,০০০ টাকা এবং করমুক্ত সীমার পরিমাণ ৩,৫০,০০০ টাকা। সকল হিসাব ও বিনিয়োগ রেবাত বিবেচনা করে আপনার নিট প্রদেয় করের পরিমাণ ১৬,০০০ টাকা নির্ধারিত হয়েছে। যেহেতু আপনার আয় করমুক্ত সীমার উপরে, তাই আপনার জন্য রিটার্ন দাখিল করা বাধ্যতামূলক।


## 📝 Step 13: Assumptions & Verification Matrix

### Key Assumptions:
1. **Taxable Income = Total Income**: No separate exemptions or deductions are subtracted before applying the tax-free threshold.
2. **Threshold Begins First**: Slabs begin strictly after the taxpayer's threshold (General: 350k, Female: 400k, Senior 65+: 400k, Disabled: 475k, Freedom Fighter: 500k).
3. **Investment Rebate**: Capped at `min(actual_investment, 20% * income) * 15%`.
4. **Minimum Tax**: Dhaka/Chattogram City (5,000 BDT), Other City Corporations (4,000 BDT), Rest of country (3,000 BDT). Net tax = `max(Gross - Rebate, Minimum Tax)` when income exceeds threshold.
5. **Source of Truth**: The local fallback DB `data/tax_db.json` is treated as ground truth if remote endpoints are unavailable.